### Exploratory Data Analysis

#### Objectives:
+ Deep understand of dataset.
+ Analysis relation between its features.
+ What factor matter the most for target columns.

#### Data Loading and Basic Overview 

In [1]:
import re 
import pandas as pd 
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt 
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import stopwords
import warnings


warnings.filterwarnings("ignore")

In [2]:
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")


print("========== Train dataset ==========")
display(train.head(3))

print("\n\n========= Test dataset ==========")
display(test.head(3))


print("\n\n========= Shape of train and test =========")
display(train.shape)

display(test.shape)

========== Train dataset ==========


,id,statement,status,loaded_at
0,31176,why do you have to think about it?,Normal,2026-07-29 14:26:13
1,26386,I know it seems like such a stupid reason to d...,Suicidal,2026-07-29 14:26:13
2,18380,every night i son and cry and promise myself t...,Depression,2026-07-29 14:26:13




========= Test dataset ==========


,id,statement,status,loaded_at
0,3008,I'm lazy to complain about it ba ihh,Normal,2026-07-29 14:26:12
1,44705,i think the wifi on my iphone is broken it wil...,Normal,2026-07-29 14:26:13
2,50186,Good tracking apps? I've been trying to find a...,Bipolar,2026-07-29 14:26:14




========= Shape of train and test =========


(42144, 4)

(10536, 4)

### Data types 

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 42144 entries, 0 to 42143
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         42144 non-null  int64
 1   statement  42144 non-null  str  
 2   status     42144 non-null  str  
 3   loaded_at  42144 non-null  str  
dtypes: int64(1), str(3)
memory usage: 1.3 MB


#### Missing values 

In [4]:
miss_value_summary = pd.DataFrame(
    {
        "train":train.isna().sum(),
        "test":test.isna().sum(),
        "train_percentage":((train.isna().sum())/(len(train))*100).round(2),
        "test_percentage":((test.isna().sum())/(len(test))*100).round(2)
    }
)
miss_value_summary.sort_values(by="train",ascending=False,inplace=True)
display(miss_value_summary)


,train,test,train_percentage,test_percentage
id,0,0,0.0,0.0
statement,0,0,0.0,0.0
status,0,0,0.0,0.0
loaded_at,0,0,0.0,0.0


There is no missing values.

### Target column:
status

In [5]:
target_colum_sum = pd.DataFrame({
    "trian":train["status"].value_counts(),
    "test":test["status"].value_counts()
})

display(target_colum_sum)

,trian,test
status,,
Normal,12994,3349
Depression,12399,3005
Suicidal,8499,2152
Anxiety,3087,754
Bipolar,2223,554
Stress,2085,502
Personality disorder,857,220


There are 7 category for mental health status. Most of the mental status have normal.

#### Analysising the statement columns

In [5]:
train['text_length'] = train['statement'].apply(
    lambda x: len(x.split())
)

test['text_length'] = test['statement'].apply(
    lambda x: len(x.split())
)

In [6]:
stat_length_sum_tr = train.groupby("status")["text_length"].describe()
stat_length_sum_ts = test.groupby("status")["text_length"].describe()

display(stat_length_sum_tr.sort_values(by="mean", ascending=False))
display(stat_length_sum_ts.sort_values(by="mean", ascending=False))


,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
Personality disorder,857.0,181.250875,239.666184,4.0,65.0,134.0,234.0,5419.0
Bipolar,2223.0,177.031489,183.691676,4.0,74.0,130.0,219.0,4804.0
Depression,12399.0,167.900637,182.085367,1.0,55.0,114.0,217.0,2612.0
Suicidal,8499.0,147.279915,186.406703,1.0,41.0,92.0,188.0,6300.0
Anxiety,3087.0,144.953677,154.700066,1.0,41.0,104.0,192.0,1592.0
Stress,2085.0,113.680096,102.326614,1.0,64.0,86.0,126.0,1320.0
Normal,12994.0,17.290673,22.757072,1.0,6.0,10.0,18.0,248.0


,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
Bipolar,554.0,172.805054,144.107071,12.0,73.00,124.5,225.00,944.0
Personality disorder,220.0,171.854545,131.651251,5.0,72.75,143.0,238.25,849.0
Depression,3005.0,168.516805,211.749587,1.0,52.00,108.0,211.00,4239.0
Suicidal,2152.0,143.098513,189.230064,3.0,43.00,92.0,179.25,5248.0
Anxiety,754.0,139.189655,143.817031,1.0,41.00,95.0,191.75,849.0
Stress,502.0,118.258964,119.618809,1.0,64.25,90.0,130.75,1606.0
Normal,3349.0,17.042998,22.812085,1.0,5.00,9.0,18.00,255.0


Here we can see that average number of words in normal status is around 17 for both. Where others stats has huge mean bipolar and Personality disorder have 172 mean. Except normal status other all status have average word length is more then 100 in both train and test data.